# Supervised Patch MLP — Spacespresso

**Standalone Colab notebook** — no local `src/` folder required. All implementation is inlined.

**Architecture**: WideResNet50 backbone → multi-scale patch features → binary MLP classifier

**Post-processing**: background suppression · multi-view fusion · multi-view consistency boost

**Data layout expected on Drive**:
```
spacepresso/
  class_01/train/good/
  class_01/train/anomaly_*/
  class_01/ground_truth_train/anomaly_*/
  class_01/test/
  ...
```

## 1 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set to your Google Drive folder containing the spacepresso dataset
DATA_ROOT  = "/content/drive/MyDrive/spacepresso"        # ← adjust if needed
OUTPUT_DIR = "/content/drive/MyDrive/spacepresso_outputs"


In [ ]:
!pip install timm -q
# torch, scipy, Pillow, tqdm are pre-installed on Colab


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SEED       = 43
IMAGE_SIZE = 224

METHOD_CONFIG = dict(
    backbone                 = "wide_resnet50_2",
    batch_size               = 8,
    out_indices              = [2, 3],
    patchsize                = 3,
    mlp_hidden               = 256,
    mlp_epochs               = 25,
    mlp_lr                   = 1e-3,
    weight_decay             = 1e-5,
    focal_gamma              = 2.0,
    focal_alpha              = None,   # auto-computed from class ratio (recommended)
    augment_anomalies        = True,
    normal_patches_per_image = 50,
    max_imbalance_ratio      = 10,
    mlp_dropout              = 0.0,
    hard_negative_mining     = False,
    sigma                    = 4.0,
    class_wise               = True,
)

NO_BACKGROUND          = True
MULTI_VIEW_FUSION      = True
MULTI_VIEW_CONSISTENCY = True
CONSISTENCY_ALPHA      = 0.5     # strength of consistency boost (0.2-1.0)

BG_DILATION            = 16
BG_THRESHOLD           = 0.20
BG_THRESHOLD_PER_CLASS = {
    "class_01": 0.40,
    "class_02": 0.30,
    "class_08": 0.40,
}

# 0.0 = use all anomalies for training; 0.2 = hold 20% out for local AP check
VAL_ANOMALY_FRACTION = 0.0
SAVE_FILE            = True

def get_bg_threshold(class_name):
    return BG_THRESHOLD_PER_CLASS.get(class_name, BG_THRESHOLD)


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


## 2 · Utilities (inlined)

In [ ]:
# ── Core utilities (inlined — no src/ folder needed) ─────────────────────────
from __future__ import annotations
import os, re, random
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
import numpy as np
from PIL import Image

# ── ImageSample ───────────────────────────────────────────────────────────────
@dataclass
class ImageSample:
    image_id:     str
    sample_id:    str
    class_name:   str
    view_id:      str
    split:        str
    image_path:   Path
    image:        object = None   # np.ndarray | None
    label:        object = None   # int | None
    mask_path:    object = None   # Path | None
    anomaly_type: object = None   # str | None

# ── Data loading ──────────────────────────────────────────────────────────────
_IMG_EXTS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

def _parse_filename(path):
    stem = Path(path).stem
    hits = list(re.finditer(r"(?i)(?:^|[_\-.])?(view\d+)(?=$|[_\-.])", stem))
    if not hits:
        return stem, stem, "view0"
    m = hits[-1]
    view_id = m.group(1).lower()
    sample_id = re.sub(r"[_\-.]+$", "",
                       (stem[:m.start()] + stem[m.end():]).strip("_-. ")) or stem
    return stem, sample_id, view_id

def _iter_images(directory):
    d = Path(directory)
    if not d.exists():
        return []
    return sorted(p for p in d.rglob("*")
                  if p.is_file() and p.suffix.lower() in _IMG_EXTS)

def _find_mask(mask_dir, image_path):
    md, stem = Path(mask_dir), Path(image_path).stem
    if not md.exists():
        return None
    for ext in _IMG_EXTS:
        c = md / f"{stem}{ext}"
        if c.exists():
            return c
    hits = sorted(p for p in md.glob(f"{stem}*")
                  if p.is_file() and p.suffix.lower() in _IMG_EXTS)
    return hits[0] if hits else None

def _make_sample(path, class_name, split, label, mask_path, anomaly_type):
    image_id, sample_id, view_id = _parse_filename(path)
    return ImageSample(image_id=image_id, sample_id=sample_id,
                       class_name=class_name, view_id=view_id, split=split,
                       image_path=Path(path), image=None, label=label,
                       mask_path=mask_path, anomaly_type=anomaly_type)

def load_train_good(data_root):
    out = []
    for cls in sorted(p.name for p in Path(data_root).glob("class_*") if p.is_dir()):
        for p in _iter_images(Path(data_root) / cls / "train" / "good"):
            out.append(_make_sample(p, cls, "train_good", 0, None, None))
    return out

def load_train_anomalies(data_root):
    out = []
    for cls in sorted(p.name for p in Path(data_root).glob("class_*") if p.is_dir()):
        for ad in sorted((Path(data_root) / cls / "train").glob("anomaly_*")):
            if not ad.is_dir():
                continue
            md = Path(data_root) / cls / "ground_truth_train" / ad.name
            for p in _iter_images(ad):
                out.append(_make_sample(p, cls, "train_anomaly", 1,
                                        _find_mask(md, p), ad.name))
    return out

def load_test(data_root):
    out = []
    for cls in sorted(p.name for p in Path(data_root).glob("class_*") if p.is_dir()):
        for p in _iter_images(Path(data_root) / cls / "test"):
            out.append(_make_sample(p, cls, "test", None, None, None))
    return out

# ── Seed ──────────────────────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    except Exception:
        pass

# ── Evaluation ────────────────────────────────────────────────────────────────
def load_sample_mask(sample, shape):
    if sample.mask_path is None or not Path(sample.mask_path).exists():
        return np.zeros(shape, dtype=np.uint8)
    with Image.open(sample.mask_path) as im:
        im = im.convert("L")
        if im.size != (shape[1], shape[0]):
            im = im.resize((shape[1], shape[0]), resample=Image.NEAREST)
        return (np.asarray(im) > 0).astype(np.uint8)

def _ap(y_true, y_score):
    y_true  = np.asarray(y_true,  dtype=np.uint8)
    y_score = np.asarray(y_score, dtype=np.float64)
    n_pos = int(np.sum(y_true))
    if n_pos == 0:
        return 0.0
    order = np.argsort(-y_score, kind="mergesort")
    ys, ss = y_true[order], y_score[order]
    idx  = np.r_[np.where(np.diff(ss))[0], y_true.size - 1]
    tps  = np.cumsum(ys)[idx]
    fps  = 1 + idx - tps
    prec = tps / np.maximum(tps + fps, 1)
    rec  = tps / n_pos
    return float(np.sum((rec - np.r_[0.0, rec[:-1]]) * prec))

def evaluate_predictions(samples, predictions):
    pt, ps, it, is_ = [], [], [], []
    for s in samples:
        pred = np.asarray(predictions[s.image_id], dtype=np.float32)
        mask = load_sample_mask(s, pred.shape)
        pt.append(mask.reshape(-1))
        ps.append(pred.reshape(-1))
        it.append(int(mask.max() > 0 or s.label == 1))
        is_.append(float(pred.max()))
    return {
        "pixel_ap": _ap(np.concatenate(pt), np.concatenate(ps)),
        "image_ap": _ap(np.asarray(it), np.asarray(is_)),
    }

# ── q8rle encoding ────────────────────────────────────────────────────────────
def float_matrix_to_q8rle(x):
    arr = np.clip(np.nan_to_num(np.asarray(x, dtype=np.float32),
                                nan=0.0, posinf=1.0, neginf=0.0), 0.0, 1.0)
    h, w = arr.shape
    flat = np.rint(arr * 255.0).astype(np.uint8).T.reshape(-1)
    runs = []
    prev, cnt = int(flat[0]), 1
    for v in flat[1:]:
        vi = int(v)
        if vi == prev:
            cnt += 1
        else:
            runs.extend([str(prev), str(cnt)])
            prev, cnt = vi, 1
    runs.extend([str(prev), str(cnt)])
    return " ".join(["q8rle", str(h), str(w)] + runs)

def q8rle_to_float_matrix(s):
    toks = [t for t in re.split(r"[\s:]+", s.strip()) if t]
    h, w = int(toks[1]), int(toks[2])
    vals = []
    for vt, rt in zip(toks[3::2], toks[4::2]):
        r = int(rt)
        if r:
            vals.append(np.full(r, int(vt), dtype=np.uint8))
    return np.concatenate(vals).reshape(w, h).T.astype(np.float32) / 255.0

def _prep_pred_map(x):
    arr = np.nan_to_num(np.asarray(x, dtype=np.float32),
                        nan=0.0, posinf=1.0, neginf=0.0)
    mn, mx = float(arr.min()), float(arr.max())
    if mn < 0.0 or mx > 1.0:
        dr = mx - mn
        arr = (arr - mn) / dr if dr > 1e-12 else np.zeros_like(arr)
    return np.clip(arr, 0.0, 1.0).astype(np.float32)

print("Utilities loaded.")


## 3 · Method (inlined)

In [ ]:
# ── Supervised Patch MLP (inlined) ───────────────────────────────────────────
# Requires: utilities cell above, plus torch/timm/scipy installed
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from scipy.ndimage import gaussian_filter

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(it, **kw): return it

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
GLOBAL_KEY    = "__global__"

# Symmetric classes: D4 augmentation (4 rotations x 2 flips = 8 transforms)
_ROTATION_4 = {"class_03", "class_05", "class_06", "class_07"}
# Oriented classes: Klein-4 (0/180 rotations x 2 flips = 4 transforms)
_ROTATION_2 = {"class_01", "class_02", "class_04", "class_08"}

def _augmentations_for_class(class_name):
    k_values = [0, 1, 2, 3] if class_name in _ROTATION_4 else [0, 2]
    return [(k, flip) for k in k_values for flip in (False, True)]

def _focal_loss(logits, targets, alpha, gamma):
    bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    pt      = torch.exp(-bce)
    alpha_t = targets * alpha + (1.0 - targets) * (1.0 - alpha)
    return (alpha_t * (1.0 - pt) ** gamma * bce).mean()

def _build_feature_extractor(backbone, out_indices, image_size, device):
    model = timm.create_model(backbone, pretrained=True, features_only=True,
                               out_indices=out_indices).to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1).to(device)
    std  = torch.tensor(IMAGENET_STD ).view(1, 3, 1, 1).to(device)
    with torch.no_grad():
        h, w  = image_size
        dummy = torch.zeros(1, 3, h, w, device=device)
        feats = model((dummy - mean) / std)
    grid_h, grid_w = feats[0].shape[-2], feats[0].shape[-1]
    feat_dim = sum(f.shape[1] for f in feats)
    print(f"Backbone: {backbone} | grid={grid_h}x{grid_w} | feat_dim={feat_dim}")
    return model, mean, std, grid_h, grid_w, feat_dim

@torch.no_grad()
def _extract_features(model, mean, std, images_tensor, patchsize, device):
    images_tensor = images_tensor.to(device, non_blocking=True)
    feats = model((images_tensor - mean) / std)
    target_hw = feats[0].shape[-2:]
    processed = []
    for feat in feats:
        feat = F.avg_pool2d(feat, kernel_size=patchsize, stride=1, padding=patchsize // 2)
        if feat.shape[-2:] != target_hw:
            feat = F.interpolate(feat, size=target_hw, mode="bilinear", align_corners=False)
        processed.append(feat)
    out = torch.cat(processed, dim=1)
    return out.permute(0, 2, 3, 1).contiguous()  # (B, H, W, C)

def _load_image_tensor(path, image_size):
    h, w = image_size
    with Image.open(path) as im:
        im  = im.convert("RGB").resize((w, h), resample=Image.BILINEAR)
        arr = np.asarray(im, dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)  # (3, H, W)

def _load_mask_at_grid(mask_path, grid_h, grid_w):
    with Image.open(mask_path) as im:
        im = im.convert("L").resize((grid_w, grid_h), resample=Image.NEAREST)
        return (np.asarray(im) > 0)  # (grid_h, grid_w) bool

class _MLP(nn.Module):
    def __init__(self, in_dim, hidden=256, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

class _PatchDataset(torch.utils.data.Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels   = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]

def _normalize_maps(raw):
    if not raw:
        return {}
    mn = min(float(np.nanmin(v)) for v in raw.values())
    mx = max(float(np.nanmax(v)) for v in raw.values())
    dr = mx - mn
    if dr <= 1e-12:
        return {k: np.zeros_like(v, dtype=np.float16) for k, v in raw.items()}
    return {k: np.clip((v.astype(np.float32) - mn) / dr, 0.0, 1.0).astype(np.float16)
            for k, v in raw.items()}


class SupervisedPatchMLP:
    def __init__(self, config):
        self.seed            = int(config.get("seed", 42))
        image_size           = config.get("image_size", 224)
        self.image_size      = (image_size, image_size) if isinstance(image_size, int) else tuple(image_size)
        self.backbone        = str(config.get("backbone", "wide_resnet50_2"))
        self.out_indices     = tuple(int(x) for x in config.get("out_indices", [2, 3]))
        self.patchsize       = int(config.get("patchsize", 3))
        self.batch_size      = int(config.get("batch_size", 8))
        self.mlp_hidden      = int(config.get("mlp_hidden", 256))
        self.mlp_epochs      = int(config.get("mlp_epochs", 20))
        self.mlp_lr          = float(config.get("mlp_lr", 1e-3))
        self.weight_decay    = float(config.get("weight_decay", 1e-5))
        self.focal_gamma     = float(config.get("focal_gamma", 2.0))
        self.focal_alpha     = config.get("focal_alpha", None)
        self.normal_patches_per_image = int(config.get("normal_patches_per_image", 50))
        self.augment_anomalies        = bool(config.get("augment_anomalies", True))
        self.max_imbalance_ratio      = int(config.get("max_imbalance_ratio", 10))
        self.mlp_dropout              = float(config.get("mlp_dropout", 0.0))
        self.hard_negative_mining     = bool(config.get("hard_negative_mining", False))
        self.hnm_epochs               = int(config.get("hnm_epochs", 10))
        self.hnm_ratio                = int(config.get("hnm_ratio", 3))
        self.sigma                    = float(config.get("sigma", 4.0))
        self.class_wise               = bool(config.get("class_wise", True))

        device_str = config.get("device") or ("cuda" if torch.cuda.is_available() else "cpu")
        self.device = torch.device(device_str)

        self.model, self.mean, self.std, self.grid_h, self.grid_w, self.feat_dim = \
            _build_feature_extractor(self.backbone, self.out_indices, self.image_size, self.device)
        self.mlps       = {}
        self.feat_norms = {}

    def fit(self, train_samples):
        grouped = self._group(train_samples) if self.class_wise else {GLOBAL_KEY: train_samples}
        for key, samples in grouped.items():
            n_anom = sum(1 for s in samples if s.label == 1)
            print(f"\nFitting MLP for {key}: {len(samples)} images ({n_anom} anomalous)")
            feats, labels, f_mean, f_std = self._build_patch_dataset(samples)
            self.feat_norms[key] = (f_mean, f_std)
            feats = (feats - f_mean) / f_std
            mlp = self._train_mlp(feats, labels, key, self.mlp_epochs)
            if self.hard_negative_mining:
                feats, labels = self._mine_hard_negatives(mlp, feats, labels)
                mlp = self._train_mlp(feats, labels, key, self.hnm_epochs,
                                      init_state=mlp.state_dict())
            self.mlps[key] = mlp
        return self

    def predict(self, test_samples):
        if not self.mlps:
            raise RuntimeError("Call fit() before predict()")
        grouped = self._group(test_samples) if self.class_wise else {GLOBAL_KEY: test_samples}
        raw = {}
        for key, key_samples in grouped.items():
            mlp_key = key if key in self.mlps else GLOBAL_KEY
            raw.update(self._predict_class(key_samples, self.mlps[mlp_key],
                                           self.feat_norms[mlp_key]))
        return _normalize_maps(raw)

    def _build_patch_dataset(self, samples):
        all_feats, all_labels = [], []
        generator = torch.Generator()
        generator.manual_seed(self.seed)

        for i in tqdm(range(0, len(samples), self.batch_size),
                      desc="Extracting patch features", leave=False):
            batch = samples[i : i + self.batch_size]
            imgs  = torch.stack([_load_image_tensor(s.image_path, self.image_size)
                                  for s in batch])
            patch_feats = _extract_features(self.model, self.mean, self.std,
                                             imgs, self.patchsize, self.device)
            B, H, W, C  = patch_feats.shape
            flat_feats   = patch_feats.reshape(B, H * W, C).cpu()

            for j, s in enumerate(batch):
                if s.label == 1 and s.mask_path:
                    mask_grid = _load_mask_at_grid(s.mask_path, H, W)
                    augments  = (_augmentations_for_class(s.class_name)
                                 if self.augment_anomalies else [(0, False)])
                    for k, do_flip in augments:
                        img_aug  = torch.rot90(imgs[j], k, dims=[1, 2])
                        mask_aug = np.rot90(mask_grid, k)
                        if do_flip:
                            img_aug  = torch.flip(img_aug, dims=[2])
                            mask_aug = np.fliplr(mask_aug)
                        aug_feats = _extract_features(
                            self.model, self.mean, self.std,
                            img_aug.unsqueeze(0), self.patchsize, self.device)
                        all_feats.append(aug_feats.reshape(H * W, C).cpu())
                        all_labels.append(torch.from_numpy(
                            mask_aug.reshape(-1).astype(np.float32)))
                else:
                    n_keep = min(self.normal_patches_per_image, H * W)
                    idx = torch.randperm(H * W, generator=generator)[:n_keep]
                    all_feats.append(flat_feats[j][idx])
                    all_labels.append(torch.zeros(n_keep, dtype=torch.float32))

        feats  = torch.cat(all_feats,  dim=0)
        labels = torch.cat(all_labels, dim=0)
        n_pos  = int(labels.sum().item())
        n_neg  = len(labels) - n_pos

        if n_pos > 0 and n_neg > n_pos * self.max_imbalance_ratio:
            n_neg_keep = n_pos * self.max_imbalance_ratio
            neg_idx  = torch.where(labels == 0)[0]
            keep     = neg_idx[torch.randperm(len(neg_idx), generator=generator)[:n_neg_keep]]
            pos_idx  = torch.where(labels == 1)[0]
            all_idx  = torch.cat([pos_idx, keep])
            feats    = feats[all_idx]
            labels   = labels[all_idx]
            n_neg    = n_neg_keep

        n_pos = int(labels.sum().item())
        print(f"  Patches: {len(labels):,} | anomalous: {n_pos:,} "
              f"({100*n_pos/max(len(labels),1):.2f}%) | ratio={n_neg//max(n_pos,1)}:1")
        normal_feats = feats[labels == 0]
        f_mean = normal_feats.mean(dim=0)
        f_std  = normal_feats.std(dim=0).clamp(min=1e-6)
        return feats, labels, f_mean, f_std

    def _train_mlp(self, feats, labels, key, epochs, init_state=None):
        mlp = _MLP(self.feat_dim, self.mlp_hidden, self.mlp_dropout).to(self.device)
        if init_state is not None:
            mlp.load_state_dict(init_state)
        optimizer = torch.optim.Adam(mlp.parameters(), lr=self.mlp_lr,
                                      weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(epochs, 1), eta_min=1e-6)

        n_pos   = float(labels.sum().item())
        n_total = float(len(labels))
        alpha   = (float(self.focal_alpha) if self.focal_alpha is not None
                   else min(0.95, 1.0 - n_pos / n_total))
        gamma   = self.focal_gamma
        print(f"  Focal loss: alpha={alpha:.3f}, gamma={gamma}")

        loader = torch.utils.data.DataLoader(
            _PatchDataset(feats, labels), batch_size=4096, shuffle=True, num_workers=0)
        best_loss, best_state = float("inf"), None

        for epoch in range(1, epochs + 1):
            mlp.train()
            epoch_loss = []
            for xb, yb in loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                optimizer.zero_grad(set_to_none=True)
                loss = _focal_loss(mlp(xb), yb, alpha=alpha, gamma=gamma)
                loss.backward()
                optimizer.step()
                epoch_loss.append(loss.item())
            scheduler.step()
            mean_loss = float(np.mean(epoch_loss))
            print(f"  [{key}] epoch {epoch:02d}/{epochs} | "
                  f"loss={mean_loss:.5f} | lr={scheduler.get_last_lr()[0]:.2e}")
            if mean_loss < best_loss:
                best_loss  = mean_loss
                best_state = {k: v.clone() for k, v in mlp.state_dict().items()}

        mlp.load_state_dict(best_state)
        mlp.eval()
        return mlp

    @torch.no_grad()
    def _mine_hard_negatives(self, mlp, feats, labels):
        mlp.eval()
        neg_idx   = torch.where(labels == 0)[0]
        neg_feats = feats[neg_idx].to(self.device)
        scores = torch.cat([
            torch.sigmoid(mlp(neg_feats[i : i + 4096]))
            for i in range(0, len(neg_feats), 4096)
        ]).cpu()
        n_hard   = int(labels.sum().item()) * self.hnm_ratio
        _, hard_rel = torch.topk(scores, min(n_hard, len(scores)))
        hard_idx = neg_idx[hard_rel]
        pos_idx  = torch.where(labels == 1)[0]
        keep     = torch.cat([pos_idx, hard_idx])
        return feats[keep], labels[keep]

    @torch.no_grad()
    def _predict_class(self, samples, mlp, feat_norm):
        predictions = {}
        f_mean, f_std = feat_norm
        f_mean = f_mean.to(self.device)
        f_std  = f_std.to(self.device)
        mlp.eval()

        for i in tqdm(range(0, len(samples), self.batch_size),
                      desc="MLP inference", leave=False):
            batch = samples[i : i + self.batch_size]
            imgs  = torch.stack([_load_image_tensor(s.image_path, self.image_size)
                                  for s in batch])
            patch_feats = _extract_features(self.model, self.mean, self.std,
                                             imgs, self.patchsize, self.device)
            B, H, W, C  = patch_feats.shape
            flat   = (patch_feats.reshape(B * H * W, C) - f_mean) / f_std
            scores = torch.sigmoid(mlp(flat)).reshape(B, H, W)
            scores_up = F.interpolate(
                scores.unsqueeze(1), size=self.image_size,
                mode="bilinear", align_corners=False).squeeze(1)

            for j, s in enumerate(batch):
                amap = scores_up[j].cpu().numpy().astype(np.float32)
                if self.sigma > 0:
                    amap = gaussian_filter(amap, sigma=self.sigma).astype(np.float32)
                predictions[str(s.image_id)] = amap.astype(np.float16)

        return predictions

    @staticmethod
    def _group(samples):
        grouped = {}
        for s in samples:
            grouped.setdefault(s.class_name, []).append(s)
        return grouped

print("SupervisedPatchMLP ready.")


## 4 · Load data

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
set_seed(SEED)

print("Loading data from:", DATA_ROOT)
train_good      = load_train_good(DATA_ROOT)
train_anomalies = load_train_anomalies(DATA_ROOT)
test            = load_test(DATA_ROOT)
print(f"train_good={len(train_good):,}  "
      f"train_anomalies={len(train_anomalies)}  test={len(test):,}")

# Optional: restrict to specific classes for fast debugging
DEV_CLASSES = None  # e.g. {"class_01", "class_03"} for quick iteration
if DEV_CLASSES:
    train_good      = [s for s in train_good      if s.class_name in DEV_CLASSES]
    train_anomalies = [s for s in train_anomalies if s.class_name in DEV_CLASSES]
    test            = [s for s in test            if s.class_name in DEV_CLASSES]
    print(f"Filtered to {DEV_CLASSES}")

# Train/val split on labeled anomalies
import random as _rng
_rng.seed(SEED)
_shuffled = _rng.sample(train_anomalies, len(train_anomalies))
_n_val = max(1, int(len(_shuffled) * VAL_ANOMALY_FRACTION)) if VAL_ANOMALY_FRACTION > 0 else 0
train_anomalies_val = _shuffled[:_n_val]
train_anomalies_fit = _shuffled[_n_val:]
print(f"Anomaly split: {len(train_anomalies_fit)} fit / {len(train_anomalies_val)} val")


## 5 · (Optional) Background mask preview

Run to visually verify that foreground masks look correct on your data.

In [ ]:
# Run this cell to visually verify that foreground masks look correct
if NO_BACKGROUND:
    import matplotlib.pyplot as plt
    import random as _r
    from scipy.ndimage import binary_fill_holes, binary_dilation as _bdilate

    def _preview_fg(image_np, threshold, dilation):
        fg = binary_fill_holes(image_np.mean(axis=2) > threshold)
        if dilation > 0:
            fg = _bdilate(fg, iterations=dilation)
        return fg.astype(np.float32)

    picked = _r.sample(train_good, min(6, len(train_good)))
    fig, axes = plt.subplots(2, len(picked), figsize=(3 * len(picked), 6))
    for col, s in enumerate(picked):
        img = np.array(Image.open(s.image_path).convert("RGB")
                       .resize((IMAGE_SIZE, IMAGE_SIZE))) / 255.0
        thr  = get_bg_threshold(s.class_name)
        mask = _preview_fg(img, thr, BG_DILATION)
        axes[0, col].imshow(img)
        axes[0, col].set_title(f"{s.class_name}\nthr={thr}", fontsize=7)
        axes[0, col].axis("off")
        axes[1, col].imshow(mask, cmap="gray")
        axes[1, col].set_title("fg mask", fontsize=7)
        axes[1, col].axis("off")
    plt.suptitle(f"Background masks (dilation={BG_DILATION})")
    plt.tight_layout()
    plt.show()


## 6 · Train

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
model = SupervisedPatchMLP({"seed": SEED, "image_size": IMAGE_SIZE, **METHOD_CONFIG})
model.fit(train_good + train_anomalies_fit)
print("Training complete.")


## 7 · Inference

In [ ]:
# ── Inference ────────────────────────────────────────────────────────────────
predictions = model.predict(test)
print(f"Predicted {len(predictions)} images.")

# Quick visualization of a few test predictions
import matplotlib.pyplot as plt
import random as _r

_shown = _r.sample(test, min(4, len(test)))
fig, axes = plt.subplots(len(_shown), 2, figsize=(8, 3 * len(_shown)))
if len(_shown) == 1:
    axes = [axes]
for row, s in zip(axes, _shown):
    img  = np.array(Image.open(s.image_path).convert("RGB")
                    .resize((IMAGE_SIZE, IMAGE_SIZE))) / 255.0
    pred = np.asarray(predictions[s.image_id], dtype=np.float32)
    row[0].imshow(img);  row[0].axis("off"); row[0].set_title(s.class_name, fontsize=8)
    row[1].imshow(pred, cmap="magma", vmin=0, vmax=1)
    row[1].axis("off"); row[1].set_title("anomaly map", fontsize=8)
plt.tight_layout()
plt.show()


## 8 · Post-processing

In [ ]:
# ── Post-processing ──────────────────────────────────────────────────────────
from collections import defaultdict
from scipy.ndimage import binary_fill_holes, binary_dilation as _bdilate

# 1. Background suppression
if NO_BACKGROUND and predictions:
    id_to_sample = {s.image_id: s for s in test}
    for img_id, pred in predictions.items():
        s   = id_to_sample[img_id]
        thr = get_bg_threshold(s.class_name)
        img = np.array(Image.open(s.image_path).convert("RGB")
                       .resize((IMAGE_SIZE, IMAGE_SIZE))) / 255.0
        fg  = binary_fill_holes(img.mean(axis=2) > thr)
        if BG_DILATION > 0:
            fg = _bdilate(fg, iterations=BG_DILATION)
        predictions[img_id] = pred * fg.astype(np.float32)
    print("Background suppression applied.")

# Build per-sample view map
sample_to_views = defaultdict(list)
for s in test:
    sample_to_views[s.sample_id].append(s.image_id)

# 2. Multi-view fusion: propagate max anomaly score to all views of same object
if MULTI_VIEW_FUSION and predictions:
    n_boosted = 0
    for image_ids in sample_to_views.values():
        maps = [predictions[iid] for iid in image_ids if iid in predictions]
        if not maps:
            continue
        sample_max = max(float(m.max()) for m in maps)
        for iid in image_ids:
            if iid not in predictions:
                continue
            view_max = float(predictions[iid].max())
            if view_max < sample_max - 1e-6:
                scale = sample_max / max(view_max, 1e-6)
                predictions[iid] = np.clip(predictions[iid] * scale, 0.0, 1.0).astype(
                    predictions[iid].dtype)
                n_boosted += 1
    print(f"Multi-view fusion: {n_boosted} views boosted.")

# 3. Multi-view consistency boost
if MULTI_VIEW_CONSISTENCY and predictions:
    n_boosted = 0
    for image_ids in sample_to_views.values():
        present = [iid for iid in image_ids if iid in predictions]
        if len(present) < 2:
            continue
        scores   = [float(predictions[iid].max()) for iid in present]
        s_max    = max(scores)
        s_mean   = float(np.mean(scores))
        if s_mean < 1e-6:
            continue
        boost = 1.0 + CONSISTENCY_ALPHA * (s_max - s_mean) / (s_max + 1e-6)
        for iid in present:
            predictions[iid] = np.clip(predictions[iid] * boost, 0.0, 1.0).astype(
                predictions[iid].dtype)
            n_boosted += 1
    print(f"Multi-view consistency: {n_boosted} views boosted (alpha={CONSISTENCY_ALPHA}).")


## 9 · Validation

Only runs when `VAL_ANOMALY_FRACTION > 0`. Set it in the config cell above.

In [ ]:
# ── Local validation (skipped when VAL_ANOMALY_FRACTION=0.0) ─────────────────
if train_anomalies_val:
    import random as _r
    import matplotlib.pyplot as plt
    from collections import defaultdict
    from scipy.ndimage import binary_fill_holes, binary_dilation as _bdilate

    N_GOOD_VAL  = 200
    good_sample = _r.sample(train_good, min(N_GOOD_VAL, len(train_good)))
    val_samples = train_anomalies_val + good_sample
    val_pred    = model.predict(val_samples)

    # Mirror the same post-processing as the test pipeline
    if NO_BACKGROUND:
        _id2s = {s.image_id: s for s in val_samples}
        for img_id, pred in val_pred.items():
            s   = _id2s[img_id]
            thr = get_bg_threshold(s.class_name)
            img = np.array(Image.open(s.image_path).convert("RGB")
                           .resize((IMAGE_SIZE, IMAGE_SIZE))) / 255.0
            fg  = binary_fill_holes(img.mean(axis=2) > thr)
            if BG_DILATION > 0:
                fg = _bdilate(fg, iterations=BG_DILATION)
            val_pred[img_id] = pred * fg.astype(np.float32)

    val_s2v = defaultdict(list)
    for s in val_samples:
        val_s2v[s.sample_id].append(s.image_id)

    if MULTI_VIEW_FUSION:
        for iids in val_s2v.values():
            maps = [val_pred[i] for i in iids if i in val_pred]
            if not maps: continue
            sm = max(float(m.max()) for m in maps)
            for i in iids:
                if i not in val_pred: continue
                vm = float(val_pred[i].max())
                if vm < sm - 1e-6:
                    val_pred[i] = np.clip(val_pred[i] * (sm / max(vm, 1e-6)),
                                          0.0, 1.0).astype(val_pred[i].dtype)

    if MULTI_VIEW_CONSISTENCY:
        for iids in val_s2v.values():
            present = [i for i in iids if i in val_pred]
            if len(present) < 2: continue
            scores = [float(val_pred[i].max()) for i in present]
            sm, smean = max(scores), float(np.mean(scores))
            if smean < 1e-6: continue
            boost = 1.0 + CONSISTENCY_ALPHA * (sm - smean) / (sm + 1e-6)
            for i in present:
                val_pred[i] = np.clip(val_pred[i] * boost, 0.0, 1.0).astype(val_pred[i].dtype)

    metrics = evaluate_predictions(val_samples, val_pred)
    print(f"Validation on {len(train_anomalies_val)} anomalies + {len(good_sample)} good:")
    print(f"  Pixel AP: {metrics['pixel_ap']:.4f}")
    print(f"  Image AP: {metrics['image_ap']:.4f}")

    # Visualize 2 anomalies per class
    grouped = {}
    for s in train_anomalies_val:
        grouped.setdefault(s.class_name, []).append(s)

    if grouped:
        n_cls = len(grouped)
        fig, axes = plt.subplots(n_cls, 6, figsize=(18, 3 * n_cls), squeeze=False)
        for row, (cls, slist) in enumerate(sorted(grouped.items())):
            picks = _r.sample(slist, min(2, len(slist)))
            for col_off, s in enumerate(picks):
                img  = np.array(Image.open(s.image_path).convert("RGB")
                                .resize((IMAGE_SIZE, IMAGE_SIZE))) / 255.0
                gt   = load_sample_mask(s, (IMAGE_SIZE, IMAGE_SIZE)).astype(np.float32)
                pred = np.asarray(val_pred[s.image_id], dtype=np.float32)
                c    = col_off * 3
                axes[row, c].imshow(img); axes[row, c].axis("off")
                axes[row, c].set_title(cls if col_off == 0 else "", fontsize=7)
                axes[row, c+1].imshow(gt,   cmap="hot", vmin=0, vmax=1)
                axes[row, c+1].axis("off")
                axes[row, c+1].set_title("gt" if row == 0 else "", fontsize=7)
                axes[row, c+2].imshow(pred, cmap="hot", vmin=0, vmax=1)
                axes[row, c+2].axis("off")
                axes[row, c+2].set_title("pred" if row == 0 else "", fontsize=7)
        plt.suptitle("Validation — anomalies", fontsize=9)
        plt.tight_layout()
        plt.show()
else:
    print("No validation set (VAL_ANOMALY_FRACTION=0.0). Skipping local evaluation.")


## 10 · Save submission

In [ ]:
# ── Save submission ───────────────────────────────────────────────────────────
if predictions and SAVE_FILE:
    import zipfile, random as _r
    from datetime import datetime
    from pathlib import Path

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir   = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_path  = out_dir / f"supervised_patch_mlp_{timestamp}.zip"
    csv_name  = f"supervised_patch_mlp_{timestamp}.csv"

    sorted_ids = sorted(predictions)
    assert len(sorted_ids) == len(test), \
        f"Expected {len(test)} predictions, got {len(sorted_ids)}"

    spot = set(_r.sample(range(len(sorted_ids)), min(5, len(sorted_ids))))
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        with zf.open(csv_name, "w") as f:
            f.write(b"ID,Label\n")
            for i, img_id in enumerate(sorted_ids):
                label = float_matrix_to_q8rle(_prep_pred_map(predictions[img_id]))
                f.write(f"{img_id},{label}\n".encode("utf-8"))
                if i in spot:
                    decoded = q8rle_to_float_matrix(label)
                    assert decoded.shape == (IMAGE_SIZE, IMAGE_SIZE), \
                        f"Shape mismatch: {decoded.shape}"

    size_mb = zip_path.stat().st_size / 1024**2
    print(f"Saved:  {zip_path}")
    print(f"Size:   {size_mb:.1f} MB")
    print(f"Rows:   {len(sorted_ids)}")
    print("Upload the zip file directly to Kaggle.")
